<a href="https://colab.research.google.com/github/Laiba-saeed92/Deep-Learning-and-NLP-Projects/blob/main/RAG_Application_using_LlamaIndex_and_MistralAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#making a RAG application using opensource framework and llama index and mistral AI
!pip install llama-index-llms-huggingface

In [ ]:
!pip install llama-index

In [ ]:
#Convert text/documents into embeddings (vectors)then store these vectors in a vector database (FAISS, Chroma, Pinecone, Weaviate, etc.)
#Later retrieve the most relevant vectors when a user asks a question then feed retrieved chunks to an LLM for answering
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.llms.huggingface import HuggingFaceLLM #This is a wrapper class that lets you use Hugging Face models as the LLM inside LlamaIndex,It gives you flexibility to plug in local or HuggingFace-hosted LLMs into the LlamaIndex pipeline.

In [ ]:
!mkdir data

mkdir: cannot create directory ‘data’: File exists


In [ ]:
documents= SimpleDirectoryReader("./data/").load_data()

In [ ]:
print(documents)

[Document(id_='63dced68-8df0-4a4e-b2f7-ae5051a4694c', embedding=None, metadata={'page_label': '1', 'file_name': 'RAG.pdf', 'file_path': '/content/data/RAG.pdf', 'file_type': 'application/pdf', 'file_size': 1662567, 'creation_date': '2025-12-10', 'last_modified_date': '2025-12-10'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='1\nRetrieval-Augmented Generation for Large\nLanguage Models: A Survey\nYunfan Gaoa, Yun Xiongb, Xinyu Gao b, Kangxiang Jia b, Jinliu Pan b, Yuxi Bic, Yi Dai a, Jiawei Sun a, Meng\nWangc, and Haofen Wang a,c\naShanghai Research Institute for Intelligent Autonomous Systems, Tongji University\nbShanghai Key Laborator

In [ ]:
from llama_index.core import PromptTemplate
#PromptTemplate allows you to create custom prompt formats.
system_prompt='''<|SYSTEM|># This is a question Answer Assistant. Your goal is to answer questions as accurate as possible based on the instructions and context provided'''
query_wrapper_prompt= PromptTemplate("<|USER|>{str_query}<|ASSISTANT|>")


In [ ]:
import torch
llm= HuggingFaceLLM(
    context_window=4096, #This tells LlamaIndex that your model can handle 4096 tokens of context.
    max_new_tokens=256, #This sets how many tokens the LLM is allowed to generate in response.256 tokens ≈ ~180 words.
    generate_kwargs={ "temperature":0.7,"do_sample":False}, #These are passed directly to the HuggingFace generation engine.
    system_prompt=system_prompt, #This is the role instruction for the model.You are an expert assistant. Answer accurately.This always runs before every query.
    query_wrapper_prompt=query_wrapper_prompt, #This wraps every question you send.
    tokenizer_name="mistralai/Mistral-7B-Instruct-v0.1", #Loads the correct tokenizer for the model Tokenizer splits sentences → tokens the model understands
    model_name="mistralai/Mistral-7B-Instruct-v0.1", #Loads the local model weights.It runs completely offline (no API key needed).
    device_map= "auto", #Automatically places model layers on:GPU (if available),CPU fallback if GPU memory is low,This helps you run large models even with 8GB–12GB GPU.
    stopping_ids=[50278, 50279, 50277, 1,0], #These are special tokens that tell the model:Stop generating text when any of these appearEnd of sequence (EOS),Stop marker tokens
    tokenizer_kwargs={"max_length":4096}, #This makes sure input sequences do not exceed 4096 tokens.If longer, tokenizer truncates gracefully.
    model_kwargs={"torch_dtype":torch.float16}



)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [ ]:
!pip install llama-index-embeddings-huggingface
!pip install llama-index-embeddings-instructor


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.5/171.5 kB 5.2 MB/s eta 0:00:00
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.1.2
    Uninstalling sentence-transformers-5.1.2:
      Successfully uninstalled sentence-transformers-5.1.2


In [ ]:
!pip install  sentence-transformers transformers faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 42.8 MB/s eta 0:00:00


In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding


embedding = HuggingFaceEmbedding(model_name="sentence-transformers/all-mpnet-base-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
index= VectorStoreIndex.from_documents(documents,embed_model=embedding)

It creates a Query Engine that:

Takes your question

Searches the vector index → finds similar chunks

Sends those chunks + your question to the LLM

LLM generates an answer using only the retrieved context

Returns a final structured response

In [ ]:
query_engine= index.as_query_engine(llm=llm) #This line wraps your vector index with an LLM, forming a “RAG pipeline”


In [ ]:
response= query_engine.query("what is RAG?")

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
